# Sprint 0: GPU Fundamentals

Why is a GPU faster than a CPU -- and when is it not? This notebook runs a
few small experiments on a Tesla T4 to find out, instead of just assuming it.

## 1. Check GPU access
Confirm the GPU is available and read its basic specs.

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
    print("SM count:", torch.cuda.get_device_properties(0).multi_processor_count)

CUDA available: True
GPU name: Tesla T4
Total VRAM (GB): 15.64
SM count: 40


## 2. CPU to GPU transfer bandwidth
A CPU and a GPU have separate memory. Before anything can run on the GPU, it
has to be copied there over PCIe. This measures how fast that copy is.

In [2]:
import time
import torch

n = 4096
x_cpu = torch.randn(n, n, dtype=torch.float32)
size_mb = x_cpu.numel() * x_cpu.element_size() / 1e6
print("Tensor device:", x_cpu.device)
print("Tensor size (MB):", round(size_mb, 1))

_ = torch.randn(10, device="cuda")  # warm-up
torch.cuda.synchronize()

times = []
for _ in range(5):
    torch.cuda.synchronize()
    start = time.perf_counter()
    x_gpu = x_cpu.to("cuda")
    torch.cuda.synchronize()
    times.append(time.perf_counter() - start)

best = min(times)
print("Tensor device now:", x_gpu.device)
print("Transfer times (ms):", [round(t * 1000, 2) for t in times])
print("Best bandwidth (GB/s):", round(size_mb / 1000 / best, 2))
print("VRAM used (MB):", round(torch.cuda.memory_allocated() / 1e6, 1))

Tensor device: cpu
Tensor size (MB): 67.1
Tensor device now: cuda:0
Transfer times (ms): [29.03, 14.55, 14.09, 14.02, 14.0]
Best bandwidth (GB/s): 4.79
VRAM used (MB): 67.1


## 3. The async timing trap
GPU operations run asynchronously -- the CPU hands off work and moves on
immediately. Timing without `torch.cuda.synchronize()` only measures how long
it took to *launch* the work, not how long the GPU actually took to finish it.

In [3]:
import time
import torch

n = 4096
A = torch.randn(n, n, device="cuda")
B = torch.randn(n, n, device="cuda")

for _ in range(3):  # warm-up
    _ = A @ B
torch.cuda.synchronize()

flops = 2 * n**3

start = time.perf_counter()
C = A @ B
t_wrong = time.perf_counter() - start

torch.cuda.synchronize()
start = time.perf_counter()
C = A @ B
torch.cuda.synchronize()
t_right = time.perf_counter() - start

print(f"Without sync: {t_wrong*1000:.3f} ms -> {flops/t_wrong/1e12:.1f} TFLOPS")
print(f"With sync:    {t_right*1000:.3f} ms -> {flops/t_right/1e12:.1f} TFLOPS")

Without sync: 0.575 ms -> 239.1 TFLOPS
With sync:    34.692 ms -> 4.0 TFLOPS


## 4. CPU vs GPU -- one matmul
A direct comparison: the same 4096x4096 matrix multiply, once on the CPU
(NumPy, using an optimized BLAS library) and once on the GPU.

In [4]:
import os, time
import numpy as np
import torch

n = 4096
flops = 2 * n**3
print("CPU cores visible:", os.cpu_count())

A_np = np.random.randn(n, n).astype(np.float32)
B_np = np.random.randn(n, n).astype(np.float32)

_ = A_np @ B_np  # warm-up
cpu_times = []
for _ in range(3):
    start = time.perf_counter()
    _ = A_np @ B_np
    cpu_times.append(time.perf_counter() - start)
t_cpu = min(cpu_times)

A = torch.from_numpy(A_np).to("cuda")
B = torch.from_numpy(B_np).to("cuda")
for _ in range(3):  # warm-up
    _ = A @ B
torch.cuda.synchronize()

gpu_times = []
for _ in range(10):
    torch.cuda.synchronize()
    start = time.perf_counter()
    _ = A @ B
    torch.cuda.synchronize()
    gpu_times.append(time.perf_counter() - start)
t_gpu = min(gpu_times)

print(f"CPU (numpy): {t_cpu*1000:.1f} ms -> {flops/t_cpu/1e9:.1f} GFLOPS")
print(f"GPU (torch): {t_gpu*1000:.1f} ms -> {flops/t_gpu/1e9:.1f} GFLOPS")
print(f"Speedup: {t_cpu/t_gpu:.1f}x")

CPU cores visible: 4
CPU (numpy): 549.6 ms -> 250.1 GFLOPS
GPU (torch): 33.9 ms -> 4056.8 GFLOPS
Speedup: 16.2x


## 5. Where does the GPU start winning?
Sweeping matrix size from 32 to 4096 to find the crossover point -- the size
where the GPU stops losing to the CPU.

In [5]:
import time
import numpy as np
import torch

def best_time(fn, repeats, sync=False):
    fn()  # warm-up
    if sync:
        torch.cuda.synchronize()
    best = float("inf")
    for _ in range(repeats):
        if sync:
            torch.cuda.synchronize()
        start = time.perf_counter()
        fn()
        if sync:
            torch.cuda.synchronize()
        best = min(best, time.perf_counter() - start)
    return best

print(f"{'n':>6} {'cpu ms':>10} {'gpu ms':>10} {'speedup':>9} {'gpu GFLOPS':>11}")
for n in [32, 64, 128, 256, 512, 1024, 2048, 4096]:
    A_np = np.random.randn(n, n).astype(np.float32)
    B_np = np.random.randn(n, n).astype(np.float32)
    A = torch.from_numpy(A_np).to("cuda")
    B = torch.from_numpy(B_np).to("cuda")

    t_cpu = best_time(lambda: A_np @ B_np, repeats=20 if n <= 1024 else 3)
    t_gpu = best_time(lambda: A @ B, repeats=50 if n <= 1024 else 10, sync=True)

    print(f"{n:>6} {t_cpu*1000:>10.4f} {t_gpu*1000:>10.4f} {t_cpu/t_gpu:>8.1f}x {2*n**3/t_gpu/1e9:>11.1f}")

     n     cpu ms     gpu ms   speedup  gpu GFLOPS
    32     0.0025     0.0250      0.1x         2.6
    64     0.0059     0.0252      0.2x        20.8
   128     0.0333     0.0284      1.2x       147.7
   256     0.1778     0.0417      4.3x       804.6
   512     1.0820     0.0699     15.5x      3839.5
  1024     8.1058     0.3579     22.6x      6000.1
  2048    65.8545     2.3317     28.2x      7368.1
  4096   543.5171    34.6508     15.7x      3966.4


## 6. Raw memory bandwidth
Matrix multiplication involves real compute. This test uses a near-trivial
operation (`y = x + 1`) to isolate pure memory bandwidth -- how fast data can
move in and out of VRAM, without much math getting in the way.

In [6]:
import time
import torch

N = 128 * 1024 * 1024  # 128M floats = ~537 MB
x = torch.randn(N, device="cuda")
y = torch.empty_like(x)

for _ in range(3):  # warm-up
    torch.add(x, 1.0, out=y)
torch.cuda.synchronize()

best = float("inf")
for _ in range(10):
    torch.cuda.synchronize()
    start = time.perf_counter()
    torch.add(x, 1.0, out=y)
    torch.cuda.synchronize()
    best = min(best, time.perf_counter() - start)

bytes_moved = 2 * N * 4
print(f"Time: {best*1000:.2f} ms")
print(f"Bandwidth: {bytes_moved/best/1e9:.1f} GB/s  ({bytes_moved/best/1e9/320*100:.0f}% of 320 GB/s peak)")
print(f"Compute: {N/best/1e9:.1f} GFLOPS")

Time: 4.66 ms
Bandwidth: 230.5 GB/s  (72% of 320 GB/s peak)
Compute: 28.8 GFLOPS


## 7. Batching reuses weights
During decoding, a GPU reads a weight matrix from VRAM to process one token
at a time. This checks what happens when it processes several tokens (a
batch) against the same weights at once.

In [7]:
import time
import torch

n = 8192
W = torch.randn(n, n, device="cuda", dtype=torch.float16)
w_bytes = n * n * 2

print(f"{'batch':>6} {'ms':>8} {'W-read GB/s':>12} {'TFLOPS':>8}")
for batch in [1, 2, 4, 8, 16, 32, 64, 128, 256]:
    X = torch.randn(n, batch, device="cuda", dtype=torch.float16)
    for _ in range(3):  # warm-up
        _ = W @ X
    torch.cuda.synchronize()

    best = float("inf")
    for _ in range(20):
        torch.cuda.synchronize()
        start = time.perf_counter()
        _ = W @ X
        torch.cuda.synchronize()
        best = min(best, time.perf_counter() - start)

    tflops = 2 * n * n * batch / best / 1e12
    print(f"{batch:>6} {best*1000:>8.3f} {w_bytes/best/1e9:>12.1f} {tflops:>8.2f}")

 batch       ms  W-read GB/s   TFLOPS
     1    0.532        252.2     0.25
     2    0.711        188.7     0.38
     4    0.709        189.2     0.76
     8    0.692        194.1     1.55
    16    0.684        196.2     3.14
    32    0.698        192.3     6.15
    64    0.716        187.4    12.00
   128    0.748        179.4    22.96
   256    1.224        109.7    28.08


## Summary
- 40 SMs, 2560 CUDA cores, 15.64 GB VRAM
- CPU-to-GPU transfer ~4.6 GB/s, VRAM bandwidth ~239 GB/s (75% of peak)
- GPU ops are async: unsynchronized timing gave a fake 231.3 TFLOPS vs a real ~4.4 TFLOPS
- CPU vs GPU matmul crossover around n=128; ~19x speedup by n=4096
- Small GPU kernels have a ~30 microsecond fixed launch overhead
- Batching reuses already-loaded weights -- 64x more work cost only ~1.32x more time
- Exact numbers above are from one specific run -- they move around 10-15% between
  runs on a shared, free GPU. The structural findings (crossover point, overhead
  floor, batching behavior) stayed consistent; the precise speedup/TFLOPS figures
  didn't.